In [50]:
# Import necessary libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv
from lifelines import CoxPHFitter

# Clinical Data
df = pd.read_csv("./data/X_train/clinical_train.csv")
df_eval = pd.read_csv("./data/X_test/clinical_test.csv")

# Molecular Data
maf_df = pd.read_csv("./data/X_train/molecular_train.csv")
maf_eval = pd.read_csv("./data/X_test/molecular_test.csv")

target_df = pd.read_csv("./data/target_train.csv")

# TODO : Uncomment for test data ??
"""
target_df_test = pd.read_csv("./data/target_test.csv")
"""
# Preview the data
df.head()

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]"
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx"
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]"
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]"
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]"


In [51]:
# Drop rows where 'OS_YEARS' is NaN if conversion caused any issues
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)

# Check the data types to ensure 'OS_STATUS' is boolean and 'OS_YEARS' is numeric
print(target_df[['OS_STATUS', 'OS_YEARS']].dtypes)

# Contarget_dfvert 'OS_YEARS' to numeric if it isn’t already
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')

# Ensure 'OS_STATUS' is boolean
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

OS_STATUS    float64
OS_YEARS     float64
dtype: object


In [52]:
# Count the number of unique values for each column
unique_values_count = maf_df.nunique()
print(unique_values_count)

ID                3026
CHR                 23
START             4645
END               4664
REF                406
ALT                332
GENE               124
PROTEIN_CHANGE    4686
EFFECT              16
VAF               2952
DEPTH             2268
dtype: int64


In [53]:
# Step: Extract the number of somatic mutations per patient
# Group by 'ID' and count the number of mutations (rows) per patient
tmp = maf_df.groupby('ID').size().reset_index(name='Nmut')

# Merge with the training dataset and replace missing values in 'Nmut' with 0
df = df.merge(tmp, on='ID', how='left').fillna({'Nmut': 0})

In [98]:
# Select features
features = ['BM_BLAST', 'HB', 'PLT', 'WBC','ANC','Nmut']
target = ['OS_YEARS', 'OS_STATUS']

# Create the survival data format
X = df.loc[df['ID'].isin(target_df['ID']), features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_df)

In [99]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [100]:
# Survival-aware imputation for missing values
imputer = SimpleImputer(strategy="median")
X_train[features] = imputer.fit_transform(X_train[features])
X_test[features] = imputer.transform(X_test[features])

In [101]:
features_train = features + ['CENTER'] + ['OS_YEARS', 'OS_STATUS']

df_train = df[features + ['ID','CENTER']].merge(target_df[['ID','OS_YEARS','OS_STATUS']], on='ID', how='left')
df_train = df_train.dropna(subset=features_train)
df_train.head()

,BM_BLAST,HB,PLT,WBC,ANC,Nmut,ID,CENTER,OS_YEARS,OS_STATUS
0,14.0,7.6,119.0,2.8,0.2,9.0,P132697,MSK,1.115068,True
1,1.0,11.6,42.0,7.4,2.4,3.0,P132698,MSK,4.928767,False
2,15.0,14.2,81.0,3.7,2.1,3.0,P116889,MSK,2.043836,False
3,1.0,8.9,77.0,3.9,1.9,11.0,P132699,MSK,2.476712,True
4,6.0,11.1,195.0,128.0,9.7,1.0,P132700,MSK,3.145205,False


In [102]:
# Initialize and train the Cox Proportional Hazards model
cox = CoxPHFitter()
cox.fit(df_train[features_train],duration_col="OS_YEARS", event_col="OS_STATUS", strata="CENTER")

# Evaluate the model using Concordance Index IPCW
cox_cindex_train = concordance_index_ipcw(y_train, y_train, cox.predict_partial_hazard(X_train))[0]
cox_cindex_test = concordance_index_ipcw(y_train, y_test, cox.predict_partial_hazard(X_test))[0]
print(f"Cox Proportional Hazard Model Concordance Index IPCW on train: {cox_cindex_train:.5f}")
print(f"Cox Proportional Hazard Model Concordance Index IPCW on test: {cox_cindex_test:.5f}")

/home/carl/.local/lib/python3.10/site-packages/lifelines/utils/__init__.py:1185: UserWarning: Attempting to convert an unexpected datatype 'object' to float. Suggestion: 1) use `lifelines.utils.datetimes_to_durations` to do conversions or 2) manually convert to floats/booleans.
  warnings.warn(warning_text, UserWarning)


Cox Proportional Hazard Model Concordance Index IPCW on train: 0.67574
Cox Proportional Hazard Model Concordance Index IPCW on test: 0.66845


In [103]:

tmp_eval = maf_eval.groupby('ID').size().reset_index(name='Nmut')


# Merge with the training dataset and replace missing values in 'Nmut' with 0
df_eval = df_eval.merge(tmp_eval, on='ID', how='left').fillna({'Nmut': 0})



In [10]:
df_eval[features] = imputer.transform(df_eval[features])

prediction_on_test_set = cox.predict(df_eval.loc[:, features])

In [11]:
submission = pd.Series(prediction_on_test_set, index=df_eval['ID'], name='risk_score')

In [12]:
submission.to_csv('./submission/scratch.csv')